# Decision Tree Classifier

## Imports

In [11]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from collections import Counter

## 1. Data Preparation

### 1.1 Load and Split Data

In [12]:
data = load_breast_cancer()
X = data.data
t = data.target

# split dataset into 70% training, 30% validation + test
X_train, X_temp, t_train, t_temp = train_test_split(X, t, test_size=0.3, random_state=42, stratify=t)
    
# split the remaining into 15% validation and 15% test sets
X_val, X_test, t_val, t_test = train_test_split(X_temp, t_temp, test_size=0.5, random_state=42, stratify=t_temp)

#### Verify Split Sizes

In [13]:
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 398 samples
Validation set size: 85 samples
Test set size: 86 samples


Since we're going to retrain again on the combined training and validation set after hyperparameter tuning, they are merged back together.

In [14]:
X_main = np.concatenate([X_train, X_val])
t_main = np.concatenate([t_train, t_val])

In [15]:
print(f"Main set size: {X_main.shape[0]} samples")

Main set size: 483 samples


## 2. Decision Tree Class Implementation

### 2.1 Node Class

Nodes are like containers that hold information about each split. It has the feature to be split on, the threshold value for the split, left and right child nodes which result from the split, and a value that is only used for leaf nodes to represent the class label.

The * in the constructor indicates that any parameters after it must be specified with their names. Generally, the value parameter is not provided, it's only for the leaf nodes.

In [16]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature      # index of feature to split on
        self.threshold = threshold  # value to split at
        self.left = left            # left child node
        self.right = right          # right child node
        self.value = value          # only for leaf nodes, class label
        
    def is_leaf_node(self):
        return self.value is not None

### 2.2 Decision Tree Class

The class is initialized with stopping criteria parameters such as minimum samples required to split a node, maximum depth of the tree.

The `fit()` method starts the main recursive `_grow_tree()` function that builds the tree. It stops when stopping criteria are met.

**Stopping Criteria:**
- Maximum tree depth (max_depth). 
- Minimum number of samples required to split a node (min_samples_split). 
- Nodes where all samples belong to the same class. 

In [17]:
class DecisionTree:
    def __init__(self, minimum_samples_split=2, max_depth=10):
        self.minimum_samples_split = minimum_samples_split
        self.max_depth = max_depth
        self.n_features = None
        self.root = None
        
    def fit(self, X, y):
        self.n_features = X.shape[1]
        self.root = self._grow_tree(X, y)
        
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))
        
        # stopping criteria
        if depth >= self.max_depth or n_labels == 1 or n_samples < self.minimum_samples_split:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        # find the best split
        feat_indices = range(n_feats)
        best_feature_index, best_threshold = self._best_split(X, y, feat_indices) 
        
        # stopping criteria if no split is found
        if best_feature_index is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        # split to left/right subtrees with the given best split
        left_indices, right_indices = self._split(X[:, best_feature_index], best_threshold)
        
        left_subtree = self._grow_tree(X[left_indices, :], y[left_indices], depth + 1)
        right_subtree = self._grow_tree(X[right_indices, :], y[right_indices], depth + 1)
        
        return Node(feature=best_feature_index, threshold=best_threshold, left=left_subtree, right=right_subtree)
    
    def _best_split(self, X, y, feat_indices):
        best_gain = -1
        split_index = None
        split_threshold = None

        # for all features we want to check (access them with their indices)
        for feat_index in feat_indices:
            X_column = X[:, feat_index]
            # get all unique values to use as potential thresholds
            thresholds = np.unique(X_column)

            # loop through all possible thresholds for this feature
            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)

                if gain > best_gain:
                    best_gain = gain
                    split_index = feat_index
                    split_threshold = threshold

        return split_index, split_threshold
    
    def _split(self, X_column, split_thresh):
        left_indices = np.argwhere(X_column <= split_thresh).flatten()
        right_indices = np.argwhere(X_column > split_thresh).flatten()
        return left_indices, right_indices

    def _most_common_label(self, y):
        counter = Counter(y)
        value = counter.most_common(1)[0][0]
        return value
        
    # Mathematical functions
    
    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_indices, right_indices = self._split(X_column, threshold)

        # No split occurred, no information gain
        if len(left_indices) == 0 or len(right_indices) == 0:
            return 0
        
        n = len(y)
        n_left = len(left_indices)
        n_right = len(right_indices)
        e_left = self._entropy(y[left_indices])
        e_right = self._entropy(y[right_indices])
        
        # Weighted average of the child entropy
        child_entropy = (n_left / n) * e_left + (n_right / n) * e_right

        information_gain = parent_entropy - child_entropy
        return information_gain
    
    def _entropy(self, y):
        # bincount returns a histogram of counts of each unique value in y
        histogram = np.bincount(y)
        
        # divide by total number to get probabilities
        ps = histogram / len(y)
    
        return -np.sum([p * np.log(p) for p in ps if p > 0])

**Note:** We don't use log of base 2 for entropy calculation here. It works fine since it's only used for comparison purposes.

Other notes:

- `argwhere()` returns indices where the condition is True, flatten is used to convert lists of lists into a single list.